# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (not as dict)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Short info
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all record sets and their field @id values
record_sets = list(dataset.record_sets)
print(f"Record sets found ({len(record_sets)}):")
rs_info = []
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']}")
    fields = rs['fields'] if 'fields' in rs else []
    if not fields:
        print('  No fields found.')
    for fld in fields:
        print(f"    * {fld['@id']}: {fld['name']} (dataType: {fld.get('dataType', 'N/A')})")
    rs_info.append({'@id': rs['@id'], 'fields': [fld['@id'] for fld in fields]})

Below, we print the first few records for each record set using their `@id`. This gives a preview of dataset structure.


In [ ]:
# Print samples of first few records for each record set
for record_set in record_sets:
    print(f"\nRecords from record set {record_set['@id']}:")
    try:
        records = dataset.records(record_set=record_set['@id'])
        for i, rec in enumerate(records):
            pprint.pprint(rec)
            if i >= 2:
                print('...')
                break
    except Exception as e:
        print(f"Could not load records for {record_set['@id']}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**NOTE**: Ensure you select the record set and field `@id`s from the overview above. Here, we will extract all record sets.

In [ ]:
# Extract data from each record set into a pandas DataFrame
dataframes = {}
for record_set in record_sets:
    rec_id = record_set['@id']
    print(f"Extracting DataFrame for {rec_id} ...")
    try:
        data = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(data)
        dataframes[rec_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"  Error extracting {rec_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
We'll perform EDA on one main record set (typically survey responses). Adjust the variable below to match the target record set `@id` and field `@id` from the overview above.

In [ ]:
# Identify a main record set for analysis
# Please copy-paste the desired record set @id as found above. If not clear, use the first listed.
main_record_set = record_sets[0]['@id'] if record_sets else None
df = dataframes[main_record_set].copy()
print(f"Analyzing DataFrame for {main_record_set}, shape: {df.shape}")
# List numeric fields in this record set
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_candidates:
    # Try to coerce object columns to numeric if possible
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields available: {numeric_candidates}")
numeric_field = numeric_candidates[0] if numeric_candidates else None
assert numeric_field is not None, "No numeric field detected for EDA analysis."

threshold = df[numeric_field].mean()  # Use the mean as threshold for illustration
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (head):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if present
group_candidates = df.select_dtypes(include=['object']).columns.tolist()
group_field = group_candidates[0] if group_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped numeric data by '{group_field}':")
    print(grouped_df[[numeric_field, f"{numeric_field}_normalized"]].head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group field
if group_field:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, preview, and analyze a Croissant-structured open dataset using the `mlcroissant` library. We reviewed available record sets and fields, extracted data, performed basic filtering and normalization for EDA, and plotted distributions for selected numeric variables. Adjust this notebook as needed to explore additional aspects or prepare the data for machine learning applications.